In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-stage-3-2026")

print("Path to dataset files:", path)

In [ ]:
import os
from torch.utils.data import Dataset,DataLoader
classes=os.listdir(os.path.join(path,'PlantVillage','train'))
print(classes)
#print(os.listdir(os.path.join(path,'PlantVillage','train','Potato___healthy')))

In [ ]:
# Write your code here
import torch
from PIL import Image
class DT(Dataset):
  def __init__(self,root,split,transform=False):
    self.full_path=os.path.join(root,'PlantVillage',split)
    self.transform=transform
    self.classes=os.listdir(self.full_path)
    self.mapper={'healthy':0,'Late_blight':1,'Early_blight':2}

    self.sampels=[]
    for label in self.classes:
      self.images_path=os.path.join(self.full_path,label)
      #self.all_path+=self.images_path[]

      for img in os.listdir(self.images_path):
        self.sampels.append((os.path.join(self.images_path,img),self.mapper[label.split('Potato___')[1]]))

  def __len__(self):
    return len(self.sampels)


  def __getitem__(self,idx):
    #img=self.sampels[idx][0]
    #target=self.sampels[idx][1]
    #img_path=os.path.join(self.images_path,img)
    image,target=self.sampels[idx]

    image=Image.open(image)

    if self.transform:
      image=self.transform(image)
    return image,target









In [ ]:
d=DT(root=path,split='train')
x,y=next(iter(d))





In [ ]:
import torchvision.transforms as transforms

transform_train = transforms.Compose([
    transforms.Resize((32,32)),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
])

transform_test = transforms.Compose([
    transforms.Resize((32,32)),
    transforms.ToTensor(),
])


train_data=DT(root=path,split='train',transform=transform_train)
test_data=DT(root=path,split='test',transform=transform_test)

train_loader=DataLoader(train_data,batch_size=32,shuffle=True,drop_last=True)
test_loader=DataLoader(test_data,batch_size=32,shuffle=False)



In [ ]:
img,target=next(iter(train_loader))
print(img.shape,target.shape)

In [ ]:
'''import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 5, figsize=(10, 5))
for i, ax in enumerate(axes.flat):
    img,_=next(iter(train_loader))
    img = img[i].squeeze(0)
    img.permute(1,2,0)

    ax.imshow(img.numpy(), cmap="gray")
    ax.axis("off")

plt.show()'''  ## here i couldent viz because i do not have time sorry

In [ ]:
# Write your code here
import torch
import torch.nn as nn

class Cnn(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(


            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128) ,
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        with torch.no_grad():  # to get the size faster
          dummy_x=torch.randn(1,3,32,32)
          feature=self.features(dummy_x)
          self.size=feature.numel()

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(self.size, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x



In [ ]:
# Write your code here
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model=Cnn(3).to(device)



In [ ]:
from tqdm import tqdm

def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in tqdm(dataloader):
        images, labels = images.to(device), labels.to(device)


        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        outputs = torch.softmax(outputs, dim=1)
        predictions = outputs.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy


def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()


            outputs = torch.softmax(outputs, dim=1)
            predictions = outputs.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy



criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
num_epochs = 10

train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

for epoch in range(num_epochs):
    train_loss, train_accuracy = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_accuracy = validate(model, test_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_accuracy)
    val_accuracies.append(val_accuracy)

    print(f"Epoch {epoch+1}/{num_epochs}: "
          f"Train Loss={train_loss:.4f}, Train Accuracy={train_accuracy:.2f}%, "
          f"Val Loss={val_loss:.4f}, Val Accuracy={val_accuracy:.2f}%")


In [ ]:
# Write your code here
